# RAG exemplo2
## documento pdf anuario de segurança publica


## Libraries

In [1]:
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
import datetime
import time
import requests
from tqdm.auto import tqdm



# # Modelos LLM (Large Language Models)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.language_models.chat_models import BaseChatModel

#embedding
from langchain_huggingface import HuggingFaceEmbeddings  



from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

from langchain_core.output_parsers import StrOutputParser


# Criação e execução de agentes
from langchain_classic.agents import( 
Tool, 
AgentExecutor,
create_tool_calling_agent,
create_react_agent)

# # Ferramentas customizadas para agentes
from langchain.tools import tool
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_experimental.tools.python.tool import PythonAstREPLTool

from langchain_classic.memory import ConversationBufferMemory


from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter,MarkdownHeaderTextSplitter

# # Componentes de RAG (Retrieval-Augmented Generation)
from langchain_chroma import Chroma  # Armazenamento vetorial


print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))



# 11/08/2026 - 15:13:32


In [2]:
# 1. Configuração do diretório de saída
OUTPUT_DOCUMENTS_DIR: str = './documentos2/'
os.makedirs(OUTPUT_DOCUMENTS_DIR, exist_ok=True)

# 2. Carregamento das variáveis de ambiente (.env)
ENV_PATH: str = '/home/akel/PycharmProjects/InsurMinds2026/.env'

def carrega_variaveis_ambiente() -> None:
    if os.path.exists(ENV_PATH):
        load_dotenv(ENV_PATH, override=True)
        print("✔ Variáveis de ambiente carregadas do arquivo .env")
    else:
        print(f"⚠ Aviso: Arquivo {ENV_PATH} não foi encontrado no diretório atual.")

print('✔ OUTPUT_DOCUMENTS_DIR:',OUTPUT_DOCUMENTS_DIR)
carrega_variaveis_ambiente()


llm_gemini = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.1-flash-lite-preview",google_api_key=os.getenv("GOOGLE_API"))

print("✔ LLM carregada:",llm_gemini.profile['name'])


✔ OUTPUT_DOCUMENTS_DIR: ./documentos2/
✔ Variáveis de ambiente carregadas do arquivo .env
✔ LLM carregada: Gemini 3.1 Flash Lite Preview


## Criando Banco de dados
### 1.leitura da fonte

In [3]:
# acessando documento 
OUTPUT_DOCUMENTS_DIR
document = PyPDFDirectoryLoader(OUTPUT_DOCUMENTS_DIR).load()
print("✔ documento carregado")
print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))


✔ documento carregado
# 11/08/2026 - 15:13:33


### 2.Splitter

In [4]:

# Splitter document
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,        # ← menor: mais preciso na recuperação
    chunk_overlap=150,     # ← overlap proporcional
    separators=["\n\n", "\n", ".", "!", "?", " "],  # ← respeita parágrafos
    length_function=len,
)

split_documents = text_splitter.split_documents(document)

for i, split in enumerate(split_documents):
    split.metadata.update({
        "chunk_id": i,
        "chunk_total": len(split_documents),
        "posicao": f"{i/len(split_documents )*100:.0f}%"  # posição no livro
    })
print("# Documento fatiado em:", time.strftime("%d/%m/%Y - %H:%M:%S"))
print(" chunk_total:", len(split_documents ))
print("--------------------")



# Documento fatiado em: 11/08/2026 - 15:13:33
 chunk_total: 54
--------------------


### 3.Embedding Model

In [8]:
model_embedding1='intfloat/multilingual-e5-small'
model_embedding2='intfloat/multilingual-e5-large-instruct'
embedding_e5= HuggingFaceEmbeddings(
    model_name=model_embedding1,
    model_kwargs={
        "device": "cpu",
        "trust_remote_code": True},
    encode_kwargs={
        "normalize_embeddings": True,
        "prompt": "passage: " })

print('#',time.strftime("%d/%m/%Y - %H:%M:%S"))

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 7604.05it/s]


# 11/08/2026 - 15:16:47


### 4.VectorStore

In [ ]:
#vectorstore = Chroma.from_documents(chunks, embedding_e5, persist_directory=f'{OUTPUT_DOCUMENTS_DIR}vectorstore')


In [7]:
# # Tamanho de cada lote
batch_size = 100
total = len(split_documents )

# Cria o banco com o primeiro lote
print(f"\n#Iniciando criação do banco: {total} chunks")
print(f"#Início: {time.strftime('%H:%M:%S')}\n")

vectorstore = Chroma.from_documents(
    documents=split_documents[:batch_size],
    embedding=embedding_e5,
    persist_directory=f'{OUTPUT_DOCUMENTS_DIR}vectorstore',
    collection_metadata={"hnsw:space": "cosine"}
)

# # Adiciona os lotes restantes com progresso
for i in tqdm(range(batch_size, total, batch_size), desc="Indexando chunks"):
    lote = split_documents[i:i + batch_size]
    vectorstore.add_documents(lote)
    print(f"  Lote {i//batch_size + 1}/{total//batch_size} | "
          f"Chunks {i}–{min(i+batch_size, total)} | "
          f"{time.strftime('%H:%M:%S')}")

print(f"\n#Banco criado com {vectorstore._collection.count()} chunks.")
print(f"#Finalizado em: {time.strftime('%H:%M:%S')}")


#Iniciando criação do banco: 54 chunks
#Início: 15:15:00



Indexando chunks: 0it [00:00, ?it/s]


#Banco criado com 54 chunks.
#Finalizado em: 15:15:06


## Carregando o banco vetorial criado

In [9]:
def carrega_banco_de_dados_vetorial(path_documentos:str) -> Chroma:
    try:
        # Usando intfloat/multilinguale5small
        embedding_query = HuggingFaceEmbeddings(
            model_name='intfloat/multilingual-e5-small',
            model_kwargs={"device": "cpu",
                          "trust_remote_code": True},
            encode_kwargs={
                "normalize_embeddings": True,
                "prompt": "query: "   })

        vectorstore = Chroma(persist_directory=path_documentos, embedding_function=embedding_query)
        if vectorstore:
            print('banco de dados carregado!')
        
        return vectorstore
    except Exception as e:
        print(f"Erro ao carregar o banco de dados vetorial: {e}")
        return None

In [10]:
vectorstore = carrega_banco_de_dados_vetorial(f'{OUTPUT_DOCUMENTS_DIR}vectorstore')
# docs = None
# retriever = vectorstore.as_retriever()
# docs = retriever.invoke("Data H")


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6923.13it/s]


banco de dados carregado!


HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-small', cache_folder=None, model_kwargs={'device': 'cpu', 'trust_remote_code': True}, encode_kwargs={'normalize_embeddings': True, 'prompt': 'query: '}, query_encode_kwargs={}, multi_process=False, show_progress=False)

## criando contexto com a base de dados

In [20]:
def busca_na_base_de_documentos(pergunta:str) -> str:
    """Use esta ferramenta para responder perguntas sobre a Data H, seus produtos como NIC, Consultoria, Cyber Segurança,
       ou qualquer informação contida na base de conhecimento. A entrada deve ser a pergunta do usuário."""
    vectorstore = carrega_banco_de_dados_vetorial(f'{OUTPUT_DOCUMENTS_DIR}vectorstore')
    contexto = None
    if vectorstore:
        retriever = vectorstore.as_retriever()
        docs = retriever.invoke(pergunta)
        contexto = "\n\n".join([doc.page_content for doc in docs])
    return contexto


## Criando agente RAG

In [17]:
def string_gemini(out_agent_exe):
    if isinstance(out_agent_exe, list) and len(out_agent_exe) > 0:
        if isinstance(out_agent_exe[0], dict) and 'text' in out_agent_exe[0]:
            return out_agent_exe[0]['text']
    
    return  out_agent_exe[0]['text']


def agente_langchain_RAG1(modelo_llm=llm_gemini) -> dict:
    ferramentas = []
    memoria = ConversationBufferMemory(memory_key="chat_history", return_messages=True, input_key="input")

    prompt = PromptTemplate(
        input_variables=["input", "context", "chat_history", "agent_scratchpad"],
        template=""" {chat_history}
                Você é um agente de IA especializado em responder perguntas de segurança publica. Responda apenas informações que estão 
                dentro do seu contexto.Jamais busque informações de outras fontes
                Contexto: {context}
                Pergunta: {input}
                {agent_scratchpad}
        """)

    agente = create_tool_calling_agent(modelo_llm,ferramentas, prompt)
    executor_do_agente = AgentExecutor(agent=agente, tools=ferramentas, memory=memoria)
    return executor_do_agente

#### Executando sem contexto

In [18]:
executor_do_agente = agente_langchain_RAG1()

pergunta1 = "comente sobre taxa de Mortes Violentas Intencionais 2024 e 2025"
contexto1 = ''
resposta1 = executor_do_agente.invoke({"input": pergunta1, "context": contexto1})
resposta1=string_gemini(resposta1['output'])

print(resposta1)
print('\n' + '=' * 10)

O contexto fornecido está vazio, portanto, não há informações disponíveis para responder à sua pergunta sobre as taxas de Mortes Violentas Intencionais de 2024 e 2025.



In [21]:
executor_do_agente = agente_langchain_RAG1()

pergunta1 = "comente sobre taxa de Mortes Violentas Intencionais 2024 e 2025"
contexto = busca_na_base_de_documentos(pergunta1)
resposta2 = executor_do_agente.invoke({"input": pergunta1, "context": contexto})
resposta2=string_gemini(resposta2['output'])
#
print(resposta2)
print('\n' + '=' * 10)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6177.22it/s]


banco de dados carregado!
Com base no contexto fornecido, a taxa de Mortes Violentas Intencionais (MVI) no Brasil apresentou a seguinte dinâmica entre 2024 e 2025:

*   **Redução:** Em 2025, o Brasil registrou uma taxa de 19,1 mortes por 100 mil habitantes, o que representa uma queda de 8,2% em relação ao ano de 2024.
*   **Contexto da queda:** Essa redução nas MVI é puxada, sobretudo, pela diminuição dos homicídios dolosos. O documento ressalta que, com exceção das mortes por intervenção policial e dos feminicídios, todas as demais subcategorias de MVI apresentaram queda, mantendo uma tendência iniciada em 2018.
*   **Fatores de influência:** Embora a tendência nacional seja de queda, o texto aponta que, em estados como Rondônia e Rio de Janeiro, as mortes decorrentes de intervenção policial (MDIP) atuaram como um fator de inflação nos dados de MVI, impulsionando a taxa nesses locais na comparação entre 2024 e 2025.



In [22]:
pergunta2 = "Quais capitais obteve destaque?"
contexto = busca_na_base_de_documentos(pergunta2)
resposta3 = executor_do_agente.invoke({"input": pergunta2, "context": contexto})
resposta3=string_gemini(resposta3['output'])
print(resposta3)
print('\n' + '=' * 10)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6627.28it/s]


banco de dados carregado!


Com base no contexto fornecido, não há informações específicas sobre o desempenho ou destaque de capitais brasileiras em relação às taxas de MVI. O texto foca na análise de municípios específicos (como Eunápolis, Maracanaú, Porto Seguro e Simões Filho) e na distribuição das taxas por regiões geográficas (Nordeste, Norte, Centro-Oeste, Sudeste e Sul).



In [28]:
pergunta3 = "Qual estado apresentou pior desempenho?"
contexto = busca_na_base_de_documentos(pergunta3)
resposta3 = executor_do_agente.invoke({"input": pergunta3, "context": contexto})
resposta3=string_gemini(resposta3['output'])
print(resposta3)
print('\n' + '=' * 10)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6033.27it/s]


banco de dados carregado!
Com base no contexto fornecido, o estado que apresentou o pior desempenho na variação da taxa de Mortes Violentas Intencionais (MVI) entre 2024 e 2025 foi o **Rio Grande do Norte**, que registrou um aumento de 19,6% na taxa.

